In [ ]:
import base64,hashlib,io,pathlib,subprocess,sys,zipfile,json,shutil
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='c591de98c4a1fddc707b8e3e586878100174523e04ae83c649eb059333a3ba2d'
root=pathlib.Path("/kaggle/working/corpus-code");root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:z.extractall(root)
subprocess.run([sys.executable,"-m","pip","install","trimesh>=4,<5","hilbertcurve>=2,<3"],check=True)
matches=list(pathlib.Path("/kaggle/input").rglob("acquisition-report.json"))
print("raw acquisition reports:",[str(p) for p in matches],flush=True)
assert len(matches)==1,"Expected exactly one mounted raw acquisition report"
raw_root=matches[0].parent
out=pathlib.Path("/tmp/kenney-64-shard-3")
subprocess.run([sys.executable,"-m","tools.prepare_kenney_catalog","--index",str(root/"artifacts/kenney-raw-pilot/candidate-index.json"),"--raw-root",str(raw_root),"--output",str(out),"--grid-size","64","--start","3","--stop","944","--stride","5","--workers","2"],cwd=root,check=True)
report=json.loads((out/"conversion-report.json").read_text())
report["source_sha256"]='c591de98c4a1fddc707b8e3e586878100174523e04ae83c649eb059333a3ba2d';(pathlib.Path("/kaggle/working")/"kenney-64-shard-3-report.json").write_text(json.dumps(report,indent=2)+"\n")
shutil.copyfile(out.parent/(out.name+".zip"),pathlib.Path("/kaggle/working")/"kenney-64-shard-3.zip")
